In [ ]:
# Week 6: Warehouse Operations & Efficiency Analysis

import os
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# 1. Load Datasets
data_path = '../data/'
branches_df = pd.read_csv(os.path.join(data_path, 'branches.csv'))
sales_header_df = pd.read_csv(os.path.join(data_path, 'sales_orders_header.csv'))
sales_lines_df = pd.read_csv(os.path.join(data_path, 'sales_orders_lines.csv'))
engineered_sales_df = pd.read_csv(os.path.join(data_path, 'engineered_sales.csv'))

# 2. Calculate Warehouse Workload & Throughput KPIs
wh_performance = engineered_sales_df.groupby('branch_id').agg(
    total_orders_processed=('order_id', 'nunique'),
    total_units_processed=('quantity', 'sum'),
    total_revenue_handled=('total_revenue', 'sum'),
    products_handled=('product_id', 'nunique')
).reset_index()

# 3. Merge Branch Master Information
wh_analysis = branches_df.merge(wh_performance, on='branch_id', how='left').fillna(0)

# 4. Calculate Efficiency Ratios & Operational Metrics
wh_analysis['avg_units_per_order'] = wh_analysis['total_units_processed'] / wh_analysis['total_orders_processed'].replace(0, np.nan)
wh_analysis['avg_revenue_per_order'] = wh_analysis['total_revenue_handled'] / wh_analysis['total_orders_processed'].replace(0, np.nan)

# Workload Balance Index (Deviation from mean workload)
mean_orders = wh_analysis['total_orders_processed'].mean()
wh_analysis['workload_variance_pct'] = ((wh_analysis['total_orders_processed'] - mean_orders) / mean_orders) * 100

# 5. Save Processed Deliverable
output_path = '../data/processed/'
os.makedirs(output_path, exist_ok=True)
wh_analysis.to_csv(os.path.join(output_path, 'warehouse_performance_summary.csv'), index=False)
print('Week 6 warehouse operations pipeline completed successfully.')